# Tutorial 04 - Exercise 04: Hospital Risk Scoring & Pivot Analytics
Computes total costs, patient age segments, composite clinical risk scores, pivot aggregations, and regional exports.


In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/hospital_original.csv")

# 1. Total Cost & Age Groups
df['TotalCost'] = df['DaysAdmitted'] * df['DailyCost']

def categorize_age(age):
    if age < 30: return "Young"
    elif 30 <= age <= 50: return "Middle"
    else: return "Senior"

df['AgeGroup'] = df['Age'].apply(categorize_age)
df.rename(columns={'DailyCost': 'DailyFee'}, inplace=True)
display(df.head())


/tmp/ipykernel_16097/1958492751.py:15: Pandas4Warning: The inplace keyword in DataFrame.rename is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.rename(columns={'DailyCost': 'DailyFee'}, inplace=True)


,City,Department,Age,DaysAdmitted,DailyFee,Satisfaction,Readmitted,TotalCost,AgeGroup
0,Alger,Cardiology,45,5,200,4.5,0,1000,Middle
1,Annaba,Neurology,60,8,300,3.8,1,2400,Senior
2,Oran,Orthopedics,30,3,150,4.2,0,450,Middle
3,Alger,Cardiology,50,7,220,4.0,0,1540,Middle
4,Oran,Neurology,40,4,280,4.1,1,1120,Middle


### 2. Clinical Risk Score Computation
Formula:
$$\text{RiskScore} = (\text{DaysAdmitted} \times 0.4) + (\text{Age} \times 0.3) + (100 - \text{Satisfaction} \times 10) \times 0.3$$


In [2]:
df['RiskScore'] = (df['DaysAdmitted'] * 0.4) + (df['Age'] * 0.3) + (100 - df['Satisfaction'] * 10) * 0.3

def categorize_risk(score):
    if score > 60: return "High"
    elif 40 <= score <= 60: return "Medium"
    else: return "Low"

df['RiskCategory'] = df['RiskScore'].apply(categorize_risk)
df.sort_values(by='RiskScore', ascending=False, inplace=True)
display(df[['City', 'Age', 'DaysAdmitted', 'Satisfaction', 'RiskScore', 'RiskCategory']])


/tmp/ipykernel_16097/983988378.py:9: Pandas4Warning: The inplace keyword in DataFrame.sort_values is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.sort_values(by='RiskScore', ascending=False, inplace=True)


,City,Age,DaysAdmitted,Satisfaction,RiskScore,RiskCategory
5,Annaba,70,6,3.5,42.9,Medium
1,Annaba,60,8,3.8,39.8,Low
7,Oran,55,9,3.9,38.4,Low
3,Alger,50,7,4.0,35.8,Low
0,Alger,45,5,4.5,32.0,Low
4,Oran,40,4,4.1,31.3,Low
2,Oran,30,3,4.2,27.6,Low
6,Alger,35,2,4.7,27.2,Low


### 3. City Summary Aggregations


In [3]:
city_summary = df.groupby('City').agg(
    AvgTotalCost=('TotalCost', 'mean'),
    AvgSatisfaction=('Satisfaction', 'mean'),
    ReadmissionRate=('Readmitted', 'mean')
).reset_index()

display(city_summary)

highest_risk = df.groupby('City')['RiskScore'].mean().idxmax()
print(f"\nHighest Clinical Risk City: {highest_risk}")


,City,AvgTotalCost,AvgSatisfaction,ReadmissionRate
0,Alger,986.666667,4.400000,0.000000
1,Annaba,1680.000000,3.650000,1.000000
2,Oran,1393.333333,4.066667,0.666667



Highest Clinical Risk City: Annaba


### 4. Data Mapping & Pivot Tables


In [4]:
df['RegionCode'] = df['City'].map({'Alger': 'DZ-16', 'Annaba': 'DZ-23', 'Oran': 'DZ-31'})
df['ReadmissionStatus'] = df['Readmitted'].map({0: 'No', 1: 'Yes'})

pivot = df.pivot_table(index='City', columns='Department', values=['TotalCost', 'Satisfaction'], aggfunc='mean')
display(pivot)

# Export processed tables
df.to_csv("../data/hospital_risk.csv", index=False)
city_summary.to_csv("../data/city_summary.csv", index=False)
print("Processed tables exported to CSV.")


Satisfaction                         TotalCost            \
Department   Cardiology Neurology Orthopedics  Cardiology Neurology   
City                                                                  
Alger               4.4       NaN         NaN  986.666667       NaN   
Annaba              NaN       3.8         3.5         NaN    2400.0   
Oran                NaN       4.0         4.2         NaN    1865.0   

                        
Department Orthopedics  
City                    
Alger              NaN  
Annaba           960.0  
Oran             450.0

Processed tables exported to CSV.
